# HelioAI — Jupyter tour

A guided tour of HelioAI inside a notebook: ask questions in natural language, get
figures inline, and keep working on the generated code in the same session.

**Before running this**

1. `pip install helioai-agent` (or `uv sync` from a clone)
2. Set one LLM provider key in `.env` — see the
   [installation guide](https://erdoganfurkan.github.io/HelioAI/installation/)
3. Build the parameter index once: `helioai index` (~10 min, 83k products)

Data access itself needs no credentials.

> Outputs are stripped from this file on purpose — run the cells to produce your own.


In [ ]:
%load_ext helioai.interfaces.jupyter_magic

import os

provider = os.environ.get("HELIOAI_LLM_PROVIDER", "azure")
print(f"Provider : {provider}")
print("Ready — use %%helioai in any cell below.")

---
## 1 — Find a parameter

You do not need to know the id. Describe the quantity.


In [ ]:
%%helioai
Which speasy parameter gives the solar wind proton density from ACE?


---
## 2 — Download and plot it

The session remembers what you just resolved, so you can refer back to it.


In [ ]:
%%helioai
Plot it for 16-17 January 2005.


---
## 3 — A plasma calculation, no download

Given values, this answers immediately — no data fetch, no sandbox.


In [ ]:
%%helioai
Plasma beta in the magnetosheath: B = 20 nT, n = 25 cm-3, T = 100 eV.


---
## 4 — Browse event catalogs

217 AMDA catalogs and timetables are available as tools, not as downloads.


In [ ]:
%%helioai
Which event catalogs cover interplanetary shocks?


---
## 5 — A multi-step request → the agent plans first

For genuinely multi-stage work — comparing missions, detecting events, statistics
over a catalog — the agent lays out a short plan before executing, then carries
on without waiting for approval. Watch for the 📋 plan card before the tool calls.


In [ ]:
%%helioai
Compare the IMF Bz measured by ACE and Wind during the 2003 Halloween storm,
and estimate the propagation delay between the two spacecraft.


---
## 6 — The same analysis across a whole catalog

This is where an agent beats downloading by hand: one request, every event.


In [ ]:
%%helioai
Run a superposed epoch analysis of the IMF Bz across the ICMEs
in the Richardson & Cane catalog for 2003-2005.


---
## 7 — Direct PlasmaPy calls, no LLM

For a quick sanity check, call the tools yourself — instant, no API call. A useful
habit: compute a reference value before asking the agent, then compare.


In [ ]:
from helioai.tools.plasmapy_tools import alfven_speed, debye_length, gyrofrequency, plasma_beta

# Typical values for three plasma regimes
regimes = [
    ("Quiet solar wind", {"B_nT": 5.0, "n_cm3": 5.0, "T_eV": 10.0}),
    ("CIR compression", {"B_nT": 20.0, "n_cm3": 15.0, "T_eV": 20.0}),
    ("Magnetosheath", {"B_nT": 20.0, "n_cm3": 25.0, "T_eV": 100.0}),
]

print(f"{'Region':<22} {'β':>6}  {'V_A (km/s)':>10}  {'f_ci (Hz)':>9}  {'λ_D (m)':>8}")
print("-" * 65)
for label, p in regimes:
    b = await plasma_beta(p["B_nT"], p["n_cm3"], p["T_eV"])
    va = await alfven_speed(p["B_nT"], p["n_cm3"])
    fci = await gyrofrequency(p["B_nT"], "proton")
    ld = await debye_length(p["n_cm3"], p["T_eV"])
    print(
        f"{label:<22} {b['beta']:>6.2f}  "
        f"{va['alfven_speed_km_s']:>10.1f}  "
        f"{fci['frequency_Hz']:>9.4f}  "
        f"{ld['debye_length_m']:>8.2f}"
    )

In [ ]:
# Power spectrum helper — demo on synthetic B data
import numpy as np

from helioai.tools.plasmapy_tools import power_spectrum

# Synthetic IMF with two injected frequencies: 0.1 Hz and 0.3 Hz
dt = 0.5  # 2 Hz cadence
t = np.arange(0, 300, dt)
B = (
    np.sin(2 * np.pi * 0.10 * t)
    + 0.5 * np.sin(2 * np.pi * 0.30 * t)
    + 0.1 * np.random.randn(len(t))
)

result = await power_spectrum(B.tolist(), dt_s=dt)
print(f"Peak frequency : {result['peak_frequency_Hz']:.3f} Hz")
print(f"Peak power     : {result['peak_power']:.3f} (arb. units)")

---
## 8 — Sessions

Conversations persist to SQLite. Browse them with `%helioai_history`, continue one
with `%helioai_resume <id>`, start over with `%helioai_session reset`.


In [ ]:
# Browse sessions for this Jupyter user
%helioai_history

In [ ]:
# Resume a previous session using the first 8 chars of its id:
# %helioai_resume abc12345

# Start fresh:
%helioai_session reset

---
## 9 — Where to go next

| Interface | How |
|---|---|
| Interactive CLI | `helioai` |
| One-shot | `helioai "your query"` |
| Jupyter | this notebook |
| Web UI | `helioai serve --web` → http://localhost:7890 |
| MCP server | `helioai-mcp`, or `--http` |

Export this session as a standalone notebook — `load_data()` becomes direct
`spz.get_data(...)`, sandbox-only helpers are stripped, and a *Methods & data
acknowledgements* cell lists every recipe and reference used:

```python
%helioai_export
```

**Links**

- Documentation: <https://erdoganfurkan.github.io/HelioAI>
- speasy: <https://github.com/SciQLop/speasy>
- PlasmaPy: <https://docs.plasmapy.org>
- AMDA: <https://amda.irap.omp.eu>
- CDAWeb: <https://cdaweb.gsfc.nasa.gov>
